<h1>4.41</h1>

In [1]:
import numpy as np

# Define the column vectors of A
a0 = np.array([3, 4])
a1 = np.array([1, 1])

# Normalize a0 to find q0
q0 = a0 / np.linalg.norm(a0)

# Project a1 onto q0 to find leg 1
leg1 = np.dot(a1, q0) / np.dot(q0, q0) * q0

# Find leg 2 by subtracting the projection from a1
leg2 = a1 - leg1

# Normalize leg2 to find q1
q1 = leg2 / np.linalg.norm(leg2)

# Verify that q0 and q1 are orthogonal and unit vectors
q0_dot_q1 = np.dot(q0, q1)
q0_magnitude = np.linalg.norm(q0)
q1_magnitude = np.linalg.norm(q1)

# Compute Q and R
Q = np.column_stack((q0, q1))
R = np.dot(Q.T, np.array([[3, 1], [4, 1]]))

# Verify that A = QR
A_reconstructed = np.dot(Q, R)


# Output
print(q0,"\n")
print(leg1,"\n")
print(leg2,"\n")
print(q1,"\n") 
print(Q,"\n") 
print(R,"\n")
print(A_reconstructed,"\n")
print(q0_dot_q1,"\n")
print(q0_magnitude,"\n")
print(q1_magnitude)


[0.6 0.8] 

[0.84 1.12] 

[ 0.16 -0.12] 

[ 0.8 -0.6] 

[[ 0.6  0.8]
 [ 0.8 -0.6]] 

[[5.00000000e+00 1.40000000e+00]
 [3.10862447e-15 2.00000000e-01]] 

[[3. 1.]
 [4. 1.]] 

5.551115123125783e-16 

1.0 

1.0


<h1>4.46</h1>

In [2]:
# Redefine the matrix A
A = np.array([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
])

# Step 1: Normalize the first column to get q0
a0 = A[:, 0]
q0 = a0 / np.linalg.norm(a0)

# Step 2: Project the second column onto q0 to start the process for q1
a1 = A[:, 1]
proj_a1_onto_q0 = np.dot(a1, q0) / np.dot(q0, q0) * q0
e1 = a1 - proj_a1_onto_q0
q1 = e1 / np.linalg.norm(e1)

# Step 3: Project the third column onto q0 and q1 to start the process for q2
a2 = A[:, 2]
proj_a2_onto_q0 = np.dot(a2, q0) / np.dot(q0, q0) * q0
proj_a2_onto_q1 = np.dot(a2, q1) / np.dot(q1, q1) * q1
e2 = a2 - proj_a2_onto_q0 - proj_a2_onto_q1
q2 = e2 / np.linalg.norm(e2)

# Now Q is formed by q0, q1, and q2 as its columns
Q = np.column_stack((q0, q1, q2))

# And R is the product of Q transpose and A
R = np.dot(Q.T, A)

# Verify that A = QR
A_reconstructed = np.dot(Q, R)

# Output the results
print(f"{q0} \n\n{q1} \n\n{q2} \n\n{Q} \n\n{R} \n\n{A_reconstructed}")

[0.70710678 0.70710678 0.        ] 

[ 0.40824829 -0.40824829  0.81649658] 

[-0.57735027  0.57735027  0.57735027] 

[[ 0.70710678  0.40824829 -0.57735027]
 [ 0.70710678 -0.40824829  0.57735027]
 [ 0.          0.81649658  0.57735027]] 

[[1.41421356e+00 7.07106781e-01 7.07106781e-01]
 [0.00000000e+00 1.22474487e+00 4.08248290e-01]
 [0.00000000e+00 1.11022302e-16 1.15470054e+00]] 

[[ 1.00000000e+00  1.00000000e+00  1.68440193e-16]
 [ 1.00000000e+00 -1.62351141e-16  1.00000000e+00]
 [ 0.00000000e+00  1.00000000e+00  1.00000000e+00]]


<h1>4.47</h1>

In [3]:
def myQR(A):
    """Classical Gram-Schmidt QR factorization."""
    A = np.asarray(A, dtype=float)
    rows, cols = A.shape
    Q = np.zeros((rows, cols))
    R = np.zeros((cols, cols))
    for j in range(cols):
        v = A[:, j].copy()
        if j:
            R[:j, j] = Q[:, :j].T @ A[:, j]
            v -= Q[:, :j] @ R[:j, j]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R


rng = np.random.default_rng(2024)
A = rng.standard_normal((4, 4))
Q, R = myQR(A)
print(f"||A - QR|| = {np.linalg.norm(A - Q @ R):.2e}")
print(f"||Q^T Q - I|| = {np.linalg.norm(Q.T @ Q - np.eye(4)):.2e}")
print(f"||R - Q^T A|| = {np.linalg.norm(R - Q.T @ A):.2e}")


||A - QR|| = 1.15e-16
||Q^T Q - I|| = 5.09e-16
||R - Q^T A|| = 1.29e-15


In [4]:
def find_full_svd_components(A):
    """SVD from the eigendecomposition of A^T A (tall matrices, m >= n)."""
    A = np.asarray(A, dtype=float)
    m, n = A.shape
    if m < n:
        raise ValueError("this demonstration expects m >= n")

    eigenvalues, eigenvectors = np.linalg.eigh(A.T @ A)
    order = np.argsort(eigenvalues)[::-1]
    V = eigenvectors[:, order]
    eigenvalues = np.clip(eigenvalues[order], 0, None)

    # A V has orthogonal columns; a QR factorization completes them to a
    # full m x m basis and recovers the singular values from R.
    Q, R = np.linalg.qr(A @ V, mode="complete")
    signs = np.ones(m)
    diag = np.sign(np.diag(R))
    diag[diag == 0] = 1
    signs[:n] = diag
    U = Q * signs
    sigma = np.abs(np.diag(R))

    Sigma = np.zeros_like(A)
    Sigma[:n, :n] = np.diag(sigma)
    return U, Sigma, V.T


m, n = 3, 2
A = np.random.default_rng(2024).standard_normal((m, n))
U, Sigma, VT = find_full_svd_components(A)
print(f"reconstruction error: {np.linalg.norm(A - U @ Sigma @ VT):.2e}")
print(f"orthogonality error:  {np.linalg.norm(U.T @ U - np.eye(m)):.2e}")


reconstruction error: 8.31e-16
orthogonality error:  3.55e-16


In [5]:
A = np.array([[1, 2, 7],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)
Q, R = myQR(A)
print(f"||A - QR|| = {np.linalg.norm(A - Q @ R):.2e}")
print(f"cond(A) = {np.linalg.cond(A):.2e}")


||A - QR|| = 0.00e+00
cond(A) = 9.47e+01
